# Geospatial Workflow Planning Agent Demo

This notebook demonstrates the `geospatial_workflow_planning_agent`, a GAS service that discovers published GAS capabilities and returns a client-side workflow plan.

The planning agent is intentionally **plan-only**. It can generate executable GAS Client Python code and a notebook skeleton, but it does not run those artifacts or call downstream GAS services itself. The client, notebook user, workflow platform, or AI orchestrator decides whether to execute the returned plan.

In this example, the planner is given a GAS Server `GetCapabilities` URL, asks that server for each agent's `DescribeAgent` document, matches available agents to a multi-step geospatial workflow, and returns the default planning artifacts:

- `interactive_workflow_graph`: HTML graph showing steps and dependencies
- `workflow_json`: machine-readable workflow plan
- `notebook_skeleton`: Jupyter notebook workflow skeleton

Additional options such as `human_readable` and `gas_client_python` can be requested explicitly with `plan_outputs`.


A Web App built purly on the Geospatial Workflow Planning Agent is available here. 

https://bpb-us-e1.wpmucdn.com/sites.psu.edu/dist/0/172103/files/2026/06/geospatial_workflow_planning_agent_app.html

## Install and Import the GAS Client SDK

The public GAS Client SDK is available from PyPI as `gas-client`. If you are running this notebook inside the repository's local environment and already have the package installed, this cell is still safe to run.


In [1]:
%pip install -q gas-client

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import HTML, IFrame, Markdown, display

from gas_client import GasClient


## Configure the GAS Server and Credentials

Use your local GAS Server, a public deployment, or an ngrok URL. The planning agent needs an LLM credential because it interprets the user goal and matches it to discovered service capabilities.

The built-in GAS agents accept either `OPENAI_API_KEY` or `GIBD_API_KEY`. Other GAS servers may document different credential requirements in their own `DescribeAgent` documents.


In [3]:
project_root = Path.cwd()
if project_root.name == "examples_for_using_gas_services":
    project_root = project_root.parent

load_dotenv(project_root / ".env")

server_url = "http://127.0.0.1:4042"

openai_api_key = os.getenv("OPENAI_API_KEY")
gibd_api_key = os.getenv("GIBD_API_KEY")

default_credentials = {}
if openai_api_key:
    default_credentials["OPENAI_API_KEY"] = openai_api_key
if not default_credentials:
    raise RuntimeError("Set OPENAI_API_KEY in the repo .env file before running this notebook.")

client = GasClient(
    server_url,
    default_credentials=default_credentials,
    timeout=900,
)

planner_agent = client.agent("geospatial_workflow_planning_agent")


## Inspect the Planning Agent Description

`DescribeAgent` explains the planner's supported parameters, output options, and plan-only behavior.


In [4]:
planner_description = planner_agent.describe()
print(planner_description["profile"]["name"])
print(planner_description["profile"]["description"])
print("\nSupported plan_outputs:")
print(json.dumps(planner_description["execute_task"]["parameters"]["plan_outputs"], indent=2))


Geospatial Workflow Planning Agent
This agent discovers GAS server capabilities, reads DescribeAgent documents, decomposes a natural-language geospatial goal into workflow steps, matches each step to suitable GAS agent services, and returns client-side workflow planning artifacts. It can generate machine-readable plans, human-readable plans, GAS Client Python code, notebook skeletons, and interactive HTML workflow graphs. It does not execute downstream GAS services.

Supported plan_outputs:
{
  "required": false,
  "default": [
    "interactive_workflow_graph",
    "workflow_json",
    "notebook_skeleton"
  ],
  "allowed_values": [
    "interactive_workflow_graph",
    "workflow_json",
    "human_readable",
    "gas_client_python",
    "notebook_skeleton"
  ],
  "description": "Selects which planning artifacts to return. The default returns the interactive workflow graph first, followed by the machine-readable workflow JSON and notebook skeleton. Markdown and Python artifacts are retur

## Build the Planning Request

The key parameter is `gas_servers`. Each item should be a GAS `GetCapabilities` URL. If this parameter is omitted, the planner uses the current server's local capability documents.

Here we explicitly provide the current server's `GetCapabilities` URL to demonstrate distributed capability discovery. The same pattern works with a remote GAS Server URL.


In [5]:
capabilities_url = f"{server_url}/?SERVICE=GAS&VERSION=1.0.0&REQUEST=GetCapabilities"

workflow_goal = (
    "Download the DEM for Richland County, South Carolina, clip it to the Richland County boundary, "
    "randomly generate 100 points within the county, extract elevation values for those points, "
    "create a histogram of the sampled elevations, and display the generated points on top of the clipped DEM."
)

planning_parameters = {
    "gas_servers": [capabilities_url],
    "plan_outputs": [
        "interactive_workflow_graph",
        "workflow_json",
        "notebook_skeleton",
    ],
    "plan_detail": "executable",
    "include_validation_steps": True,
    "max_steps": 12,
}

print(capabilities_url)


http://127.0.0.1:4042/?SERVICE=GAS&VERSION=1.0.0&REQUEST=GetCapabilities


## Execute the Planning Task in Streaming Mode

Streaming mode is useful because the planner reports discovery, matching, artifact generation, and completion progress while it works.


In [6]:
planning_result = planner_agent.run_streaming_task(
    workflow_goal,
    parameters=planning_parameters,
    timeout=900,
)

[18:00:36] stream_connected: Streaming connection established.
[18:00:36] Geospatial Workflow Planning Agent: I received your request.
[18:00:36] Geospatial Workflow Planning Agent: I need to inspect the request text and any provided dataset references before running the agent. I found 0 dataset reference(s).
[18:00:36] Geospatial Workflow Planning Agent: I found the required credentials and can start the model-backed workflow.
[18:00:36] task_accepted: Task accepted. Starting streaming execution.
[18:00:37] Geospatial Workflow Planning Agent: Next I will start the workflow with the prepared inputs.
[18:00:37] Geospatial Workflow Planning Agent: I will discover GAS capabilities, match agents to workflow steps, and return a plan for client-side execution.
[18:00:37] Geospatial Workflow Planning Agent: I am discovering GAS servers and reading their published agent capabilities.
[18:00:37] Geospatial Workflow Planning Agent: Capability discovery is complete. I will use the DescribeAgent i

In [7]:
client.print_artifacts(planning_result)

Artifacts: 3
1. Workflow Plan
   role             : workflow_plan_file
   format           : json
   type             : downloadable_file
   name             : geospatial_workflow_planning_agent-4687-nkca-5708.json
   original_filename: download_the_422111.json
   size_bytes       : 14682
   url              : http://127.0.0.1:4042/agents/geospatial_workflow_planning_agent/data/geospatial_workflow_planning_agent-4687-nkca-5708.json
2. Notebook Skeleton
   role             : notebook_skeleton_file
   format           : ipynb
   type             : downloadable_file
   name             : geospatial_workflow_planning_agent-5402-cpwc-2221.ipynb
   original_filename: download_the_334994.ipynb
   size_bytes       : 15604
   url              : http://127.0.0.1:4042/agents/geospatial_workflow_planning_agent/data/geospatial_workflow_planning_agent-5402-cpwc-2221.ipynb
3. Interactive Workflow Graph
   role             : interactive_workflow_graph_file
   format           : html
   type           

In [14]:
#Download and open the HTML file in a web browser to get a better view of the interactive workflow graph
client.display_artifacts(planning_result,format="html")   

In [11]:
client.display_artifacts(planning_result,format="json")

In [12]:
client.display_artifacts(planning_result,format="ipynb")

## Use the Generated Plan

The planning agent has not executed the workflow. The next step is for a client, notebook user, workflow platform, or AI orchestrator to review the returned plan and optionally run the generated GAS Client Python or notebook skeleton.

Typical review checklist:

- Confirm each step selected the expected GAS agent.
- Confirm credentials required by each selected agent are available.
- Confirm dependencies and artifact passing are reasonable.
- Confirm validation checks are included for CRS, raster/vector overlap, join keys, and expected artifact formats.
- Run the generated code only after reviewing server URLs and credential placeholders.
